# 🔒 Day 02a: Concurrency Control — Locks, 2PL, MVCC, Timestamp Ordering

---

## 🎯 What You'll Master Today
- Lock types: Shared (S) vs Exclusive (X)
- Two-Phase Locking (2PL) protocol
- MVCC (Multi-Version Concurrency Control)
- Timestamp-based ordering
- Concurrency anomalies: dirty read, non-repeatable read, phantom

---

## 🏗️ The Library Book Analogy

> **Shared Lock (S)**: Multiple people can READ the same book simultaneously.
> **Exclusive Lock (X)**: Only ONE person can WRITE in the book — no one else can even read.
> **MVCC**: Everyone gets their own photocopy of the book — read your version, writer updates original.

```
╔══════════════════════════════════════════════════════════════════╗
║              LOCK COMPATIBILITY MATRIX                           ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║              │ Requesting S  │ Requesting X                      ║
║  ────────────┼───────────────┼──────────────                     ║
║  Holding S   │    ✓ GRANT    │   ✗ WAIT                         ║
║  Holding X   │    ✗ WAIT     │   ✗ WAIT                         ║
║  No Lock     │    ✓ GRANT    │   ✓ GRANT                        ║
║                                                                  ║
║  S+S = OK  │  S+X = WAIT  │  X+X = WAIT                        ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Lock Manager Simulation
# ============================================

class LockManager:
    def __init__(self):
        self.locks = {}  # resource → {"type": S/X, "holders": set}
        self.wait_queue = {}  # resource → [(txn, type)]
    
    def request(self, txn, resource, lock_type):
        if resource not in self.locks:
            self.locks[resource] = {"type": lock_type, "holders": {txn}}
            print(f"  ✓ {txn} acquired {lock_type}-lock on {resource}")
            return True
        
        current = self.locks[resource]
        # S+S compatible
        if current["type"] == "S" and lock_type == "S":
            current["holders"].add(txn)
            print(f"  ✓ {txn} acquired {lock_type}-lock on {resource} (shared with {current['holders'] - {txn}})")
            return True
        
        # All other combos → WAIT
        print(f"  ✗ {txn} WAITS for {lock_type}-lock on {resource} (held by {current['holders']} as {current['type']})")
        self.wait_queue.setdefault(resource, []).append((txn, lock_type))
        return False
    
    def release(self, txn, resource):
        if resource in self.locks:
            self.locks[resource]["holders"].discard(txn)
            print(f"  🔓 {txn} released lock on {resource}")
            if not self.locks[resource]["holders"]:
                del self.locks[resource]
                # Grant to waiting txn
                if resource in self.wait_queue and self.wait_queue[resource]:
                    next_txn, next_type = self.wait_queue[resource].pop(0)
                    self.request(next_txn, resource, next_type)

lm = LockManager()
print("  === Lock Compatibility Demo ===")
lm.request("T1", "row_42", "S")  # Granted
lm.request("T2", "row_42", "S")  # Granted (S+S OK)
lm.request("T3", "row_42", "X")  # WAIT (S+X conflict)
print()
lm.release("T1", "row_42")
lm.release("T2", "row_42")        # T3 gets lock now

In [ ]:
# ============================================
# 2. Two-Phase Locking (2PL)
# ============================================

print("""
  ╔════════════════════════════════════════════════════════╗
  ║           TWO-PHASE LOCKING (2PL)                     ║
  ╠════════════════════════════════════════════════════════╣
  ║                                                        ║
  ║  Phase 1: GROWING     │ Acquire locks, never release  ║
  ║  ─── Lock Point ───   │ (maximum locks held)          ║
  ║  Phase 2: SHRINKING   │ Release locks, never acquire  ║
  ║                                                        ║
  ║  Locks                                                 ║
  ║  held   ╱╲                                             ║
  ║    │   ╱  ╲                                            ║
  ║    │  ╱    ╲                                           ║
  ║    │ ╱      ╲                                          ║
  ║    │╱        ╲                                         ║
  ║    └──────────→ time                                   ║
  ║    grow  shrink                                        ║
  ║                                                        ║
  ║  Variants:                                             ║
  ║  • Basic 2PL: may deadlock                             ║
  ║  • Strict 2PL: hold X-locks until COMMIT (no cascading)║
  ║  • Rigorous 2PL: hold ALL locks until COMMIT           ║
  ║                                                        ║
  ╚════════════════════════════════════════════════════════╝
""")

# Simulate 2PL
print("  === Strict 2PL Demo ===")
class TwoPhaseTxn:
    def __init__(self, name):
        self.name = name
        self.locks = []
        self.phase = "growing"
    
    def acquire(self, resource, lock_type):
        if self.phase != "growing":
            print(f"  ✗ {self.name}: Can't acquire in shrinking phase!")
            return
        self.locks.append((resource, lock_type))
        print(f"  ✓ {self.name}: acquired {lock_type}({resource}) [phase: {self.phase}]")
    
    def commit(self):
        self.phase = "shrinking"
        print(f"  {self.name}: COMMIT — releasing all {len(self.locks)} locks")
        for r, t in self.locks:
            print(f"    🔓 release {t}({r})")
        self.locks.clear()

t1 = TwoPhaseTxn("T1")
t1.acquire("row_A", "X")
t1.acquire("row_B", "S")
t1.acquire("row_C", "X")
t1.commit()
t1.acquire("row_D", "S")  # Should fail

In [ ]:
# ============================================
# 3. MVCC (Multi-Version Concurrency Control)
# ============================================

print("  MVCC: Readers don't block writers, writers don't block readers!\n")

class MVCCDatabase:
    def __init__(self):
        self.versions = {}  # key → [(value, txn_id, timestamp)]
        self.next_txn = 1
    
    def begin_txn(self):
        txn_id = self.next_txn
        self.next_txn += 1
        print(f"  BEGIN TXN-{txn_id} (snapshot at txn_id={txn_id})")
        return txn_id
    
    def write(self, key, value, txn_id):
        self.versions.setdefault(key, []).append((value, txn_id))
        print(f"  TXN-{txn_id} WRITE {key}={value} (new version created)")
    
    def read(self, key, txn_id):
        # Read the latest version visible to this transaction
        versions = self.versions.get(key, [])
        visible = [(v, t) for v, t in versions if t <= txn_id]
        if visible:
            value, writer = visible[-1]
            print(f"  TXN-{txn_id} READ {key}={value} (written by TXN-{writer})")
            return value
        print(f"  TXN-{txn_id} READ {key}=NULL (no visible version)")
        return None

mvcc = MVCCDatabase()

# Initial data
t0 = mvcc.begin_txn()  # TXN-1
mvcc.write("balance", 1000, t0)
print()

# T2 reads, T3 writes, T2 reads again (snapshot isolation)
t2 = mvcc.begin_txn()  # TXN-2
t3 = mvcc.begin_txn()  # TXN-3

mvcc.read("balance", t2)    # Sees 1000
mvcc.write("balance", 800, t3)  # T3 creates new version
mvcc.read("balance", t2)    # Still sees 1000! (snapshot isolation)
mvcc.read("balance", t3)    # Sees 800 (its own write)

print("\n  All versions:", mvcc.versions)
print("  💡 MVCC: T2's snapshot doesn't see T3's write → no blocking!")

In [ ]:
# ============================================
# 4. Concurrency Anomalies Summary
# ============================================

print("""
  ┌───────────────────────┬─────────────────────────────────────────────────────┐
  │ Anomaly               │ Description                                         │
  ├───────────────────────┼─────────────────────────────────────────────────────┤
  │ Dirty Read            │ Read uncommitted data from another txn              │
  │                       │ T1 writes X=5. T2 reads X=5. T1 rollbacks → T2 bad │
  ├───────────────────────┼─────────────────────────────────────────────────────┤
  │ Non-Repeatable Read   │ Same row, different values in same txn              │
  │                       │ T1 reads X=5. T2 updates X=10, commits. T1 reads 10│
  ├───────────────────────┼─────────────────────────────────────────────────────┤
  │ Phantom Read          │ New rows appear/disappear in same query             │
  │                       │ T1: SELECT WHERE age>20 → 5 rows. T2 inserts. 6 rows│
  ├───────────────────────┼─────────────────────────────────────────────────────┤
  │ Lost Update           │ Two txns read same value, both write → one lost     │
  │                       │ T1,T2: read X=10. T1: X=15. T2: X=12 → T1's lost! │
  └───────────────────────┴─────────────────────────────────────────────────────┘

  Technique Comparison:
  ┌──────────────────┬────────────────┬───────────────────────┐
  │ Technique         │ Pros           │ Cons                  │
  ├──────────────────┼────────────────┼───────────────────────┤
  │ 2PL               │ Serializable   │ Deadlocks, blocking  │
  │ MVCC              │ No read blocks │ Storage overhead      │
  │ Timestamp Order   │ No deadlocks   │ More aborts           │
  │ Optimistic (OCC)  │ High throughput│ Aborts on conflict    │
  └──────────────────┴────────────────┴───────────────────────┘
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | S-lock vs X-lock? | S: shared read (S+S OK). X: exclusive write (X+anything = WAIT) |
| 2 | What is 2PL? | Growing phase (acquire) → lock point → shrinking phase (release) |
| 3 | How does MVCC work? | Each txn sees a snapshot. Writers create new versions. Readers don't block |
| 4 | 2PL vs MVCC? | 2PL: strict serializable but blocking. MVCC: concurrent reads, more storage |
| 5 | What is a lost update? | Two txns read same value, both write back → one write is lost |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • S+S = OK, S+X = WAIT, X+X = WAIT                    │
## │  • 2PL: grow → lock point → shrink (guarantees serial.) │
## │  • Strict 2PL: hold X-locks until COMMIT                │
## │  • MVCC: snapshot per txn, no reader-writer blocking    │
## │  • Know the 4 anomalies: dirty/non-repeatable/phantom   │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Isolation Levels** — Read Uncommitted → Serializable